In [5]:
import random
import math
import numpy as np
import pandas as pd
from collections import Counter

from sklearn.datasets import load_diabetes

In [6]:
data = load_diabetes()

X = data.data.tolist()
y = data.target.tolist()

In [7]:
df = pd.DataFrame(data.data, columns=data.feature_names)
df["target"] = data.target

df.head()

,age,sex,bmi,bp,s1,s2,s3,s4,s5,s6,target
0,0.038076,0.050680,0.061696,0.021872,-0.044223,-0.034821,-0.043401,-0.002592,0.019907,-0.017646,151.0
1,-0.001882,-0.044642,-0.051474,-0.026328,-0.008449,-0.019163,0.074412,-0.039493,-0.068332,-0.092204,75.0
2,0.085299,0.050680,0.044451,-0.005670,-0.045599,-0.034194,-0.032356,-0.002592,0.002861,-0.025930,141.0
3,-0.089063,-0.044642,-0.011595,-0.036656,0.012191,0.024991,-0.036038,0.034309,0.022688,-0.009362,206.0
4,0.005383,-0.044642,-0.036385,0.021872,0.003935,0.015596,0.008142,-0.002592,-0.031988,-0.046641,135.0


In [8]:
threshold = np.mean(y)

y = [1 if val > threshold else 0 for val in y]

In [9]:
class Node:

    def __init__(self, feature_index=None, threshold=None, left=None, right=None, value=None):

        self.feature_index = feature_index
        self.threshold = threshold
        self.left = left
        self.right = right
        self.value = value


class DecisionTree:

    def __init__(self, max_depth=None, min_samples_split=2):

        self.max_depth = max_depth
        self.min_samples_split = min_samples_split
        self.root = None


    def build_tree(self, X, y):

        self.root = self._grow_tree(X, y, 0)


    def _grow_tree(self, X, y, depth):

        n_samples = len(y)

        if (self.max_depth is not None and depth >= self.max_depth) \
            or n_samples < self.min_samples_split \
            or len(set(y)) == 1:

            return Node(value=self._majority_class(y))

        feature, threshold = self._best_split(X, y)

        if feature is None:
            return Node(value=self._majority_class(y))

        left_X, left_y, right_X, right_y = self._split(X, y, feature, threshold)

        left_child = self._grow_tree(left_X, left_y, depth + 1)
        right_child = self._grow_tree(right_X, right_y, depth + 1)

        return Node(feature, threshold, left_child, right_child)


    def _best_split(self, X, y):

        best_gini = float("inf")
        best_feature = None
        best_threshold = None

        n_features = len(X[0])

        for feature in range(n_features):

            values = sorted(set(row[feature] for row in X))

            for threshold in values:

                left_X, left_y, right_X, right_y = self._split(X, y, feature, threshold)

                if len(left_y) == 0 or len(right_y) == 0:
                    continue

                gini = self._weighted_gini(left_y, right_y)

                if gini < best_gini:
                    best_gini = gini
                    best_feature = feature
                    best_threshold = threshold

        return best_feature, best_threshold


    def _split(self, X, y, feature, threshold):

        left_X, left_y, right_X, right_y = [], [], [], []

        for i in range(len(X)):
            if X[i][feature] <= threshold:
                left_X.append(X[i])
                left_y.append(y[i])
            else:
                right_X.append(X[i])
                right_y.append(y[i])

        return left_X, left_y, right_X, right_y


    def _gini(self, labels):

        counts = Counter(labels)

        impurity = 1.0
        total = len(labels)

        for count in counts.values():
            p = count / total
            impurity -= p ** 2

        return impurity


    def _weighted_gini(self, left_y, right_y):

        total = len(left_y) + len(right_y)

        return (len(left_y)/total)*self._gini(left_y) + (len(right_y)/total)*self._gini(right_y)


    def _majority_class(self, y):

        return Counter(y).most_common(1)[0][0]


    def predict(self, x):

        return self._traverse_tree(x, self.root)


    def _traverse_tree(self, x, node):

        if node.value is not None:
            return node.value

        if x[node.feature_index] <= node.threshold:
            return self._traverse_tree(x, node.left)

        return self._traverse_tree(x, node.right)

In [10]:
class RandomForest:

    def __init__(self, n_trees=10, max_depth=None, min_samples_split=2, max_features=None):

        self.n_trees = n_trees
        self.max_depth = max_depth
        self.min_samples_split = min_samples_split
        self.max_features = max_features

        self.trees = []
        self.feature_subsets = []


    def build_forest(self, X, y):

        n_samples = len(X)
        n_features = len(X[0])

        if self.max_features is None:
            self.max_features = int(math.sqrt(n_features))

        for _ in range(self.n_trees):

            indices = [random.randint(0, n_samples - 1) for _ in range(n_samples)]

            X_sample = [X[i] for i in indices]
            y_sample = [y[i] for i in indices]

            feature_indices = random.sample(range(n_features), self.max_features)

            X_subset = [[row[i] for i in feature_indices] for row in X_sample]

            tree = DecisionTree(self.max_depth, self.min_samples_split)

            tree.build_tree(X_subset, y_sample)

            self.trees.append(tree)
            self.feature_subsets.append(feature_indices)


    def predict(self, x):

        predictions = []

        for tree, feature_indices in zip(self.trees, self.feature_subsets):

            x_subset = [x[i] for i in feature_indices]

            pred = tree.predict(x_subset)

            predictions.append(pred)

        return Counter(predictions).most_common(1)[0][0]

In [11]:
tree = DecisionTree(max_depth=5, min_samples_split=5)
tree.build_tree(X, y)

forest = RandomForest(n_trees=20, max_depth=5, min_samples_split=5)
forest.build_forest(X, y)

In [12]:
def accuracy(model, X, y):

    correct = 0

    for i in range(len(X)):
        if model.predict(X[i]) == y[i]:
            correct += 1

    return correct / len(X)

In [13]:
tree_acc = accuracy(tree, X, y)
forest_acc = accuracy(forest, X, y)

print("Decision Tree Accuracy:", tree_acc)
print("Random Forest Accuracy:", forest_acc)

Decision Tree Accuracy: 0.8393665158371041
Random Forest Accuracy: 0.8506787330316742


In [22]:
toke=127
sample = X[toke]

print("Decision Tree Prediction:", tree.predict(sample))
print("Random Forest Prediction:", forest.predict(sample))
print("Actual Label:", y[toke])

Decision Tree Prediction: 0
Random Forest Prediction: 0
Actual Label: 0
